In [ ]:
import json
import torch
import torch.nn as nn

from pathlib import Path

from transformers import ViTHybridImageProcessor, ViTHybridForImageClassification
from datasets import load_dataset, load_from_disk
from torchvision.transforms.functional import to_pil_image

In [ ]:
DATASET_PATH = Path("/home/sulcm/datasets/milk10k/milk10k")

MODEL_PATH = Path("/home/sulcm/models/melanet/google_hybrid_cnn_vit/melanet_debug")
# MODEL_PATH = Path("/home/sulcm/models/melanet/google_hybrid_cnn_vit/base")

# Download model base
- Modify model config for given task
    - Set correct mapping for `id2label` and `label2id`
    - Override `classifier` layer with correctly initialized `Linear` layer (in_features keep currect, out_features new number of labels)

In [ ]:
# feature_extractor = ViTHybridImageProcessor.from_pretrained("google/vit-hybrid-base-bit-384")
# model = ViTHybridForImageClassification.from_pretrained("google/vit-hybrid-base-bit-384")

In [ ]:
# model.classifier = nn.Linear(
#     in_features=model.classifier.in_features,
#     out_features=len(labels),
#     bias=True
# )

# model.config.id2label = id2label
# model.config.label2id = label2id
# model.config.model_type = "vit_hybrid"

In [ ]:
# feature_extractor.save_pretrained("/home/sulcm/models/melanet/google_hybrid_cnn_vit/base")
# model.save_pretrained("/home/sulcm/models/melanet/google_hybrid_cnn_vit/base")

# Load custom model

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

In [ ]:
feature_extractor = ViTHybridImageProcessor.from_pretrained(MODEL_PATH)
model = ViTHybridForImageClassification.from_pretrained(MODEL_PATH).eval().to(device)

In [ ]:
# dataset = load_dataset("imagefolder", data_dir=DATASET_PATH)
dataset = load_from_disk(dataset_path=DATASET_PATH)

labels = dataset["train"].features["label"].names
id2label: dict[str, str] = {}
label2id: dict[str, int] = {}
for id, label in enumerate(labels):
    id2label[id] = label
    label2id[label] = id

dataset

In [ ]:
example = dataset["train"][1]
image = example["image"]
print(image.size)
image

In [ ]:
inputs = feature_extractor(images=image, return_tensors="pt").to(device)
outputs = model(**inputs)
logits = outputs.logits
# model predicts one of the 1000 ImageNet classes
predicted_class_idx = logits.argmax(-1).item()
print("Predicted class:", model.config.id2label[predicted_class_idx])
print("Ground truth:", model.config.id2label[example["label"]])

In [ ]:
temp10 = dataset["train"][:100]

In [ ]:
inputs = feature_extractor(images=temp10["image"], return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(**inputs)
logits = outputs.logits

predicted_class_idx = logits.argmax(-1).cpu().numpy()
torch.cuda.empty_cache()

print("ID\tPred\tGT")
print("-"*21)
for idx, pred_label in enumerate(predicted_class_idx):
    print(f"{idx+1}\t{model.config.id2label[pred_label]}\t{temp10["label"][idx]}")